# Sales Brief 系統 DB 銅質層參考（`app_sales_*`）

- 用途：記錄 MI3.0 業務填報系統（PostgreSQL `MICenter.public`）同步到 Unity Catalog 的 5 張 `app_sales_*` 表的結構、關聯、
  已知的 schema 變更，以及「某月各品牌有沒有送出」的標準查法。本 repo 後續要讀這幾張表時以此為準，不要再猜欄位。
- 層級：`app_` ＝ 其他系統原樣複製（見 `docs/conventions.md` 2.1）；Sales Brief 報表 job 讀的是下游金質表 `g_sales_brief_*`。
- 輸入 table：`{catalog}.{schema}.app_sales_brand` / `app_sales_entry_submission` / `app_sales_numeric_entry` /
  `app_sales_market_intelligence` / `app_sales_entry_revision`
- 輸出 table：無（唯讀參考 notebook，只 `display`）
- 參數（widgets）：`catalog`、`schema`、`period`（yyyymm，要檢查的填報月份）
- 負責人 / 更新日期：2026-10-27（依 202610 填報狀況查核整理）

## 1. 來源與同步

| 項目 | 值 |
|------|----|
| 來源 | Azure PostgreSQL `mi30-pgsqlsrv.postgres.database.azure.com`，DB `MICenter`，schema `public` |
| 同步方式 | Databricks notebook 以 JDBC 整表讀取 → `write.mode("overwrite").option("overwriteSchema","true").saveAsTable` |
| 目標 | `micenter.mi3_datahub_prod.app_sales_*`（表名同來源） |
| 增量 | 無，每次全表覆蓋；`overwriteSchema` 是為了跟著來源的欄位變更（見 §4） |

> 同步 notebook 目前把 PG 帳密寫在 cell 內。依本 repo 規則憑證只能走 `dbutils.secrets.get(scope, key)`，
> 若要把同步 notebook 納入 repo，先改成 secret 再放進來。

## 2. 五張表的角色與關聯

| 表 | 角色 | 主鍵 / 關聯 |
|----|------|-------------|
| `app_sales_brand` | 品牌主檔。一列＝一條產品線上的一個品牌（Dell 在 MNT 與 NB 是兩列） | `id`；`app_code`（TV / NB / MNT）、`brand_code`、`is_active`、`sort_order` |
| `app_sales_entry_submission` | 送出主表。一列＝某品牌某填報月的一次送出 | `id`；`brand_master_id → app_sales_brand.id`；`fill_year`、`fill_month`、`status`、`is_deleted` |
| `app_sales_numeric_entry` | 數值事實表（年度 BP、月度 Panel Buy、庫存週數…） | 以 `submission_id` 掛在送出主表（欄位名待 DESCRIBE 確認，見 [c02]） |
| `app_sales_market_intelligence` | 市場情報文字（市場展望 / 產品技術 / 價格 / 其他） | 同上 |
| `app_sales_entry_revision` | 草稿版本表。填了沒按送出的內容在這裡 | 同上 |

```
app_sales_brand (id, app_code, brand_code)
      ▲ brand_master_id
app_sales_entry_submission (id, fill_year, fill_month, status, is_deleted)
      ▲ submission_id
app_sales_numeric_entry / app_sales_market_intelligence / app_sales_entry_revision
```

## 3. 讀表規則（必守）

1. **產品線一律從 `app_sales_brand.app_code` 取**，用 `submission.brand_master_id = brand.id` 接。
   不要用 `submission.product_line`，也不要只用 `brand_code`（Dell / HP / Asus / Samsung 同時存在於兩條產品線）。
2. 送出主表永遠加 `is_deleted = false`；要「已送出」加 `status = 'submitted'`。
3. 一次送出帶**整年 12 個月**的月度值（`target_month` 1~12），不是只有當月與未來兩個月。
   所以「每個 target_month 取最新一筆」等於最新一次送出會蓋掉該品牌整年的值；YTD 會隨業務回填而變動。
4. 判斷「某月有沒有填」以送出主表為準（§5 的查法），不要從數值表反推：數值表沒列可能是沒填、也可能是送了但該項目空的。

## 4. 已知 schema 變更

| 時點 | 變更 | 影響 |
|------|------|------|
| 2026-08 填報起 | `app_sales_entry_submission.product_line` 不再填值（全為 NULL），改以 `brand_master_id` 指到品牌主檔；主表同時多了 `app_code`、`data_sources`、`data_source_other_text` | 2026-07 以前的列 `product_line` 有值、`brand_master_id` 也有；用 `brand_master_id` 接可同時涵蓋新舊列。`JOIN ... USING (product_line, brand_code)` 在 2026-08 之後會全部接不上（NULL 不等於 NULL） |

[c03] 可重跑確認變更時點。

## 5. 「某月各品牌有沒有送出」標準查法

[c05]。名單用品牌主檔（`is_active = true`），LEFT JOIN 該月送出；三種狀態：`已送出` / `有紀錄未送出(status)` / `未填`。
2026-10-27 對 202610 的結果見本 notebook 最後一節：11 個品牌已送出，MNT Samsung、NB Dell、NB Acer 未填（最後送出 202609）。

## 6. 與下游的對應

- `g_sales_brief_numeric_entry`：`app_code` 即品牌主檔的 `app_code`；`brand_name` 為 `'<app_code>_<brand_code>'`（例 `MNT_Asus`）；
  `status`、`fill_year`、`fill_month`、`submission_id`、`submitted_at` 沿用送出主表；`target_year` / `target_month` 為數值所屬月份；
  年度 BP（`category='panel_buy', metric='yearly_bp_current'`）的 target 期＝填報期。
- `g_sales_brief_brand_summary` / `g_sales_brief_app_summary`：由市場情報文字經 LLM 彙整，key 為 `app_code`、`brand_code`、`fill_year`、`fill_month`。
- 讀這些金質表的 job：`jobs/sales_report_mail/job_sales_brief_report_v2.ipynb`，其 [c02] 的缺填判斷與 §5 等價（改以 `g_` 表的年度 BP 判斷）。


In [ ]:
# [c01] params
# 預設值請對照 config/project.yml；job 執行時由 job parameters 覆蓋。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("period", "202610")   # 要檢查的填報月份 yyyymm

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
period = int(dbutils.widgets.get("period"))
assert catalog and schema, "catalog / schema 不可為空"

T_BRAND = f"{catalog}.{schema}.app_sales_brand"
T_SUB = f"{catalog}.{schema}.app_sales_entry_submission"
T_NUM = f"{catalog}.{schema}.app_sales_numeric_entry"
T_MI = f"{catalog}.{schema}.app_sales_market_intelligence"
T_REV = f"{catalog}.{schema}.app_sales_entry_revision"
print(T_BRAND, T_SUB, T_NUM, T_MI, T_REV, period, sep="\n")


In [ ]:
# [c02] describe_tables
# 五張表的欄位。來源 schema 變更時（同步 job 用 overwriteSchema）重跑這格，並更新本 notebook 最後一節的紀錄。
for t in [T_BRAND, T_SUB, T_NUM, T_MI, T_REV]:
    print(f"\n── {t} ──")
    display(spark.sql(f"DESCRIBE TABLE {t}"))


In [ ]:
# [c03] schema_change_timeline
# 各填報月 product_line 有值 / 為 NULL 的送出筆數：確認 §4 的變更時點（預期 2026-08 起全為 NULL）。
display(spark.sql(f"""
    SELECT fill_year, fill_month,
           SUM(CASE WHEN product_line IS NULL THEN 1 ELSE 0 END)     AS null_product_line,
           SUM(CASE WHEN product_line IS NOT NULL THEN 1 ELSE 0 END) AS has_product_line,
           SUM(CASE WHEN brand_master_id IS NULL THEN 1 ELSE 0 END)  AS null_brand_master_id,
           COUNT(*) AS n
    FROM {T_SUB}
    WHERE is_deleted = false
    GROUP BY fill_year, fill_month
    ORDER BY fill_year DESC, fill_month DESC
"""))


In [ ]:
# [c04] brand_master
# 啟用中的品牌名單（＝報表的品牌 universe 候選）。
display(spark.sql(f"""
    SELECT id AS brand_master_id, app_code, brand_code, brand_name, product_line_type, sort_order,
           panel_buy_label, sell_in_label
    FROM {T_BRAND}
    WHERE is_active = true AND module_code = 'dsbgSalesEntry'
    ORDER BY app_code, sort_order, brand_code
"""))


In [ ]:
# [c05] completeness_by_brand
# 指定 period 各品牌送出狀態：已送出 / 有紀錄未送出(status) / 未填；last_period＝最後一次 submitted 的月份。
df_comp = spark.sql(f"""
    WITH roster AS (
        SELECT id AS brand_master_id, app_code, brand_code, sort_order
        FROM {T_BRAND}
        WHERE is_active = true AND module_code = 'dsbgSalesEntry'
    ),
    sub AS (
        SELECT brand_master_id, fill_year * 100 + fill_month AS period, status, submitted_at, id AS submission_id
        FROM {T_SUB}
        WHERE is_deleted = false
    ),
    this_month AS (
        SELECT brand_master_id,
               COUNT(*) AS n_sub, MAX(status) AS status_raw, MAX(submitted_at) AS submitted_at
        FROM sub WHERE period = {period}
        GROUP BY brand_master_id
    ),
    last_sub AS (
        SELECT brand_master_id, MAX(period) AS last_period
        FROM sub WHERE status = 'submitted'
        GROUP BY brand_master_id
    )
    SELECT r.app_code, r.brand_code,
           CASE WHEN t.status_raw = 'submitted'      THEN '已送出'
                WHEN t.brand_master_id IS NOT NULL   THEN CONCAT('有紀錄未送出(', t.status_raw, ')')
                ELSE '未填' END AS fill_status,
           t.n_sub, t.submitted_at, l.last_period
    FROM roster r
    LEFT JOIN this_month t USING (brand_master_id)
    LEFT JOIN last_sub   l USING (brand_master_id)
    ORDER BY r.app_code, r.sort_order, r.brand_code
""")
display(df_comp)
missing = [f"{r['app_code']} {r['brand_code']}" for r in df_comp.collect() if r["fill_status"] != "已送出"]
print(f"{period} 未送出 / 未填：{missing or '無'}")


In [ ]:
# [c06] submission_sample
# 指定 period 的送出主表原始列（含所有欄位），看資料長相用；最多 50 列。
display(spark.sql(f"""
    SELECT s.*, b.app_code AS brand_app_code
    FROM {T_SUB} s
    LEFT JOIN {T_BRAND} b ON b.id = s.brand_master_id
    WHERE s.is_deleted = false AND s.fill_year * 100 + s.fill_month = {period}
    ORDER BY b.app_code, s.brand_code
    LIMIT 50
"""))


## 附錄：2026-10-27 查核紀錄（period = 202610）

### A. `app_sales_entry_submission` DESCRIBE

| col_name | data_type |
|---|---|
| id | bigint |
| brand_master_id | int |
| emp_no | varchar(50) |
| module_code | varchar(50) |
| brand_code | varchar(50) |
| product_line | varchar(50)（2026-08 起為 NULL） |
| fill_year | smallint |
| fill_month | smallint |
| status | varchar(20)（已送出＝`submitted`） |
| submitted_at | timestamp |
| is_deleted | boolean |
| created_at / updated_at | timestamp |
| created_by / updated_by | varchar(50) |
| app_code | varchar(20) |
| data_source_other_text | varchar(500) |
| data_sources | array<string> |

### B. `app_sales_brand` DESCRIBE

| col_name | data_type |
|---|---|
| id | int |
| module_code | varchar(50)（`dsbgSalesEntry`） |
| brand_code | varchar(50) |
| brand_name | varchar(100) |
| product_line_type | varchar(50) |
| is_active | boolean |
| sort_order | int |
| created_at / updated_at | timestamp |
| created_by / updated_by | varchar(50) |
| app_code | varchar(20)（TV / NB / MNT） |
| panel_buy_label | varchar(100) |
| sell_in_label | varchar(100) |

`app_sales_numeric_entry`、`app_sales_market_intelligence`、`app_sales_entry_revision` 尚未 DESCRIBE；跑 [c02] 後補到這裡。

### C. [c03] 變更時點

| fill_year | fill_month | null_product_line | has_product_line |
|---|---|---|---|
| 2026 | 10 | 11 | 0 |
| 2026 | 9 | 14 | 0 |
| 2026 | 8 | 14 | 0 |
| 2026 | 7 | 0 | 10 |
| 2026 | 6 | 0 | 14 |
| 2026 | 5 | 2 | 8 |
| 2026 | 4 | 0 | 4 |
| 2026 | 3 | 0 | 8 |
| 2026 | 2 | 0 | 4 |
| 2026 | 1 | 0 | 4 |
| 2025 | 12 | 0 | 4 |

### D. [c05] 202610 送出狀態

| app_code | brand_code | fill_status | submitted_at | last_period |
|---|---|---|---|---|
| MNT | Dell | 已送出 | 2026-10-20 14:26 UTC | 202610 |
| MNT | HP | 已送出 | 2026-10-20 10:20 UTC | 202610 |
| MNT | Samsung | **未填** | - | 202609 |
| MNT | Asus | 已送出 | 2026-10-22 13:17 UTC | 202610 |
| NB | HP | 已送出 | 2026-10-22 08:57 UTC | 202610 |
| NB | Lenovo | 已送出 | 2026-10-22 10:16 UTC | 202610 |
| NB | Dell | **未填** | - | 202609 |
| NB | Asus | 已送出 | 2026-10-19 10:36 UTC | 202610 |
| NB | Acer | **未填** | - | 202609 |
| NB | MSI | 已送出 | 2026-10-22 10:35 UTC | 202610 |
| TV | Samsung | 已送出 | 2026-10-19 13:54 UTC | 202610 |
| TV | Hisense | 已送出 | 2026-10-22 14:01 UTC | 202610 |
| TV | TCL | 已送出 | 2026-10-21 16:37 UTC | 202610 |
| TV | TPV | 已送出 | 2026-10-22 09:45 UTC | 202610 |

每個已送出品牌各 1 筆、無未送出草稿；對照下游 `g_sales_brief_numeric_entry`，11 個已送出品牌的 10 月數值都帶年度 BP、
12 個月的月度 Panel Buy、庫存週數，`g_sales_brief_brand_summary` / `g_sales_brief_app_summary` 亦齊。
